> 💻 **This notebook runs locally — embedded mode.** `pip install figuard`; enforcement runs in-process against a local SQLite file (`~/.figuard/figuard.db`). No server, no API key, no network. Same rules as the server, held in lockstep by a shared conformance suite.
>
> **Three ways to run FiGuard — same API, same rules. Swap only the client constructor:**
>
> - 💻 **Embedded** _(this notebook)_ — `FiGuardClient()` — one app/agent, zero infra.
> - ☁️ **Free sandbox** — `FiGuardClient(base_url="https://figuard-sandbox-g1ha.onrender.com", api_key="sb_live_demo")` — quick hosted evaluation (not for production; may be wiped; no SLA).
> - 🏢 **Self-hosted server** — `FiGuardClient(base_url="https://your-figuard", api_key="...")` — budgets shared across many agents/processes; your data, your infra. [Self-host →](https://github.com/figuard/figuard-core#self-hosting)
>
> Fleet features — delegation, category allocations, shadow mode, cross-process budgets — require the sandbox or a server.

In [ ]:
# @title Step 1 — Install and configure FiGuard
import sys
!pip install --upgrade "figuard[openai-agents]>=0.3.0" openai-agents -q
for _mod in list(sys.modules.keys()):
    if _mod.startswith("figuard"):
        del sys.modules[_mod]
import figuard
print(f"✓ FiGuard {figuard.__version__} + OpenAI Agents ready")
import uuid
USER_ID = f"demo_{uuid.uuid4().hex[:8]}"
print(f"Your session ID: {USER_ID}  (labels your budgets locally)")

In [ ]:
from agents import Agent, Runner, function_tool
from figuard.integrations.openai_agents import guarded_function_tool
from figuard import FiGuardClient

client = FiGuardClient()

budget = client.create_budget(
    user_id=USER_ID,
    total_limit=500.00,
    currency="USD",
    expires_in="24h",
    authorization_expiry_seconds=300,
    intent_context="travel booking session",
)

print(f"✓ Budget created: {budget.id}")
print(f"  Total limit: ${budget.total_limit}")

session_token = budget.session_token

In [ ]:
# Embedded mode has no web UI — get_spend_tree() *is* the spend view (the same data a dashboard renders).
def show_spend_tree(client, budget_id):
    b = client.get_budget(budget_id)
    tree = client.get_spend_tree(budget_id)
    unit = b.currency or b.unit or ""
    print(f"\nSpend tree — {b.quantity_spent}/{b.total_limit} {unit} spent, "
          f"{b.available_quantity} available  ({tree.total_events} events)")
    glyph = {"CONFIRMED": "✓", "AUTHORIZED": "⏳", "DENIED": "✗", "FAILED": "⚠", "VOIDED": "↩"}
    def walk(node, depth=1):
        e = node.event
        amt = e.confirmed_quantity if e.confirmed_quantity is not None else e.requested_quantity
        label = e.description or e.entity_id or e.action_type
        label = f"{label} — " if label else ""
        print("   " * depth + f"{glyph.get(e.decision, '•')} {label}{amt} {e.currency or ''} [{e.decision}]")
        for ch in node.children:
            walk(ch, depth + 1)
    for r in tree.roots:
        walk(r)


In [ ]:
# guarded_function_tool wraps the raw function BEFORE @function_tool converts
# it to a tool schema — decorator order matters.
#
# Step 1: define the guarded callable (holds a reference we can call directly)
@guarded_function_tool(
    client=client,
    session_token=session_token,
    category="flight",
    amount_key="amount",
    agent_id="travel_agent",
)
def _book_flight(description: str, amount: float) -> str:
    """Book a flight for the given description and amount."""
    return f"Flight booked: {description} for ${amount:.2f}"

# Step 2: wrap in @function_tool for the OpenAI Agents SDK
book_flight = function_tool(_book_flight)

agent = Agent(
    name="travel_agent",
    instructions="Help the user book flights within their budget.",
    tools=[book_flight],
)

# Full agent run (requires OPENAI_API_KEY):
# result = Runner.run_sync(agent, "Book me a flight from SFO to JFK for $270")
# print(result.final_output)

# ── Demo: call the guarded function directly — no OpenAI key needed ──────────
print("Simulating two tool calls from the agent:\n")

# Call 1: within budget → should authorize and confirm
r1 = _book_flight(description="SFO → JFK roundtrip", amount=270.0)
print(f"Tool call 1: {r1}")

# Call 2: over remaining budget → should be denied
r2 = _book_flight(description="SFO → LHR business class", amount=450.0)
print(f"Tool call 2: {r2}")

# Show final budget state
b = client.get_budget(budget.id)
print(f"\nBudget: ${b.quantity_spent:.2f} spent, ${b.available_quantity:.2f} remaining of ${b.total_limit:.2f}")
show_spend_tree(client, budget.id)